# YOLOv5s Fine-tuning for Pool Guard

This notebook implements the 3-phase fine-tuning strategy to preserve person detection while improving child/adult classification.

**Key Features:**
- Low learning rates to prevent catastrophic forgetting
- Progressive layer unfreezing
- Moderate data augmentation

**⚠️ Important:** This is for FINE-TUNING, not initial training. Use this after you have:
1. Collected data using `--collect_data` flag
2. Labeled the collected images
3. Run `fine_tune_setup.py` to prepare the dataset

In [ ]:
import torch
from ultralytics import YOLO
import os

# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Configuration

In [ ]:
# Dataset path (use fine_tune_setup.py output)
DATA_YAML = r"training_data/fine_tune/data.yaml"

# Model paths - Start from pre-trained YOLOv5s (NOT your existing model)
PRETRAINED_MODEL = "yolov5s.pt"  # Or use your existing: "models/YOLOV5S_child_adult.pt"

# Training configuration
IMG_SIZE = 640
BATCH_SIZE = 16  # Adjust based on GPU memory (8 for smaller GPUs)
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
WORKERS = 0  # Windows compatibility

# Project settings
PROJECT = "runs/detect"
PHASE1_NAME = "pool_guard_phase1_frozen"
PHASE2_NAME = "pool_guard_phase2_partial"
PHASE3_NAME = "pool_guard_phase3_full"

print(f"✅ Configuration loaded")
print(f"   Dataset: {DATA_YAML}")
print(f"   Device: {DEVICE}")
print(f"   Batch Size: {BATCH_SIZE}")

## Phase 1: Frozen Backbone Training (Epochs 1-20)

**Strategy:** Freeze first 10 layers, train only classification head with low learning rate (0.001)

In [ ]:
# Load pre-trained model
model = YOLO(PRETRAINED_MODEL)
print("🚀 Model loaded: YOLOv5s")

# Phase 1: Freeze backbone (first 10 layers)
print("\n📌 Phase 1: Training with frozen backbone (epochs 1-20)...")

results_phase1 = model.train(
    data=DATA_YAML,
    epochs=20,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    
    # Learning rate (10x lower than default)
    lr0=0.001,           # Initial learning rate
    lrf=0.01,            # Final learning rate (cosine annealing)
    
    # Freeze first 10 layers (backbone)
    freeze=10,
    
    # Optimizer
    optimizer='AdamW',    # AdamW is more stable for fine-tuning
    
    # Loss weights
    box=7.5,              # Bounding box loss
    cls=0.5,              # Classification loss (lower for 2-class)
    
    # Data augmentation (moderate)
    hsv_h=0.015,          # Hue (±1.5%)
    hsv_s=0.7,            # Saturation
    hsv_v=0.4,            # Value
    degrees=5,            # Rotation (±5 degrees, conservative)
    translate=0.1,        # Translation (10%)
    scale=0.5,            # Scaling (0.5-1.5x)
    shear=2,              # Shear (±2 degrees)
    perspective=0.0,       # Perspective (disabled)
    flipud=0.0,           # Vertical flip (disabled)
    fliplr=0.5,           # Horizontal flip (50%)
    mosaic=0.5,           # Mosaic (50%)
    mixup=0.0,            # Mixup (disabled)
    
    # Training settings
    patience=20,          # Early stopping
    project=PROJECT,
    name=PHASE1_NAME,
    exist_ok=True,
    
    # Validation
    val=True,
    plots=True
)

print("\n✅ Phase 1 Complete!")
print(f"Best model saved to: {PROJECT}/{PHASE1_NAME}/weights/best.pt")

In [ ]:
# Load Phase 1 best model
phase1_model_path = f"{PROJECT}/{PHASE1_NAME}/weights/best.pt"
model = YOLO(phase1_model_path)
print("🚀 Phase 1 model loaded")

print("\n📌 Phase 2: Partial unfreezing (first 5 layers frozen, epochs 21-35)...")

results_phase2 = model.train(
    data=DATA_YAML,
    epochs=15,            # Epochs 21-35 (15 more epochs)
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    
    # Lower learning rate for Phase 2
    lr0=0.0005,           # Half of Phase 1
    lrf=0.01,
    
    # Freeze first 5 layers only
    freeze=5,
    
    # Same optimizer and settings
    optimizer='AdamW',
    box=7.5,
    cls=0.5,
    
    # Same augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5,
    translate=0.1,
    scale=0.5,
    shear=2,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.0,
    
    patience=20,
    project=PROJECT,
    name=PHASE2_NAME,
    exist_ok=True,
    
    val=True,
    plots=True
)

print("\n✅ Phase 2 Complete!")
print(f"Best model saved to: {PROJECT}/{PHASE2_NAME}/weights/best.pt")

In [ ]:
# Load Phase 2 best model
phase2_model_path = f"{PROJECT}/{PHASE2_NAME}/weights/best.pt"
model = YOLO(phase2_model_path)
print("🚀 Phase 2 model loaded")

print("\n📌 Phase 3: Full fine-tuning (all layers unfrozen, epochs 36-50)...")

results_phase3 = model.train(
    data=DATA_YAML,
    epochs=15,            # Epochs 36-50 (15 more epochs)
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    
    # Very low learning rate for Phase 3
    lr0=0.0001,           # 10x lower than Phase 1
    lrf=0.01,
    
    # No freezing - train all layers
    freeze=0,             # 0 = no freezing
    
    # Same optimizer and settings
    optimizer='AdamW',
    box=7.5,
    cls=0.5,
    
    # Same augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5,
    translate=0.1,
    scale=0.5,
    shear=2,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.0,
    
    patience=20,
    project=PROJECT,
    name=PHASE3_NAME,
    exist_ok=True,
    
    val=True,
    plots=True
)

print("\n✅ Phase 3 Complete!")
print(f"Best model saved to: {PROJECT}/{PHASE3_NAME}/weights/best.pt")
print("\n🎉 Fine-tuning complete!")
print("\nNext steps:")
print("1. Evaluate the final model (run next cell)")
print("2. Export to OpenVINO for Pi 5")
print("3. Test on real pool camera feeds")

## Model Evaluation

Validate the final model performance

In [ ]:
# Load final model
final_model_path = f"{PROJECT}/{PHASE3_NAME}/weights/best.pt"
model = YOLO(final_model_path)

# Validate
results = model.val(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    device=DEVICE
)

print("\n📊 Final Model Metrics:")
print(f"mAP@0.5: {results.box.map50:.4f}")
print(f"mAP@0.5:0.95: {results.box.map:.4f}")
print(f"\nPer-class results:")
for i, class_name in enumerate(model.names.values()):
    print(f"  {class_name}: mAP@0.5 = {results.box.maps[i]:.4f}")

## Export for Raspberry Pi 5

Export the model to OpenVINO format for optimal Pi 5 performance

In [ ]:
# Export to OpenVINO (recommended for Pi 5)
final_model_path = f"{PROJECT}/{PHASE3_NAME}/weights/best.pt"
model = YOLO(final_model_path)

export_path = model.export(
    format='openvino',
    imgsz=IMG_SIZE,
    half=True,           # FP16 for faster inference
    simplify=True
)

print(f"\n✅ Model exported to: {export_path}")
print("\nTo use on Pi 5:")
print("1. Copy the .xml and .bin files to Pi 5")
print("2. Install OpenVINO: pip install openvino")
print("3. Update main.py to use OpenVINO model")